# 18 · `gl_engine/assertions.py`

## What this file is for

**Load-time assertions. They fail. None of them warns.**

Before any rating happens, the engine can check that ISO's content is internally consistent — that every jurisdiction resolves, that declared parents exist, that no numeric column holds something non-numeric, that zero factors appear only where a zero is known to be a sentinel.

A warning would be read once and ignored. These raise.

**Depends on:** [`05-resolve-resolver`](05-resolve-resolver.ipynb), [`07-erc-tables`](07-erc-tables.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine import assertions

for name, obj in vars(assertions).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != assertions.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Run the shallow set — the deep set scans every package and takes about a minute and a half.

In [ ]:
from gl_engine import EditionResolver, assertions

resolver = EditionResolver()
report = assertions.run_all(resolver, "20260811", deep=False)

print("ok       :", report.ok)
print("failures :", len(report.failures))
print()
for check in report.checks[:8] if hasattr(report, "checks") else []:
    print("  ", check)

## The interesting case

### What each check actually asserts

In [ ]:
for line in (assertions.__doc__ or "").splitlines()[:28]:
    print(line)

### A report is data, and then it raises

In [ ]:
print("report.ok        :", report.ok)
print("report.failures  :", report.failures)
print()
print("raise_if_failed() is the bit that makes them assertions rather than notes:")
report.raise_if_failed()
print("  ...returned cleanly, because nothing failed.")

## What it refuses

The engine will not proceed past a failed load-time check. Simulate one.

In [ ]:
from gl_engine.errors import AssertionFailure

bad = assertions.Report(asof="20260811")
# Report.add builds the Check for you -- pass its fields, not a Check
bad.add("A0", "a deliberately failing check", False,
        "so you can see what a failure looks like")
try:
    bad.raise_if_failed()
except AssertionFailure as e:
    print("AssertionFailure raised:")
    print("  ", str(e)[:200])

## Try it yourself

1. Run with `deep=True`. It takes about 95 seconds — what does it check that the shallow pass doesn't?
2. Run the assertions at two different dates. Does anything differ?
3. Find the sentinel check. Which tables are allowed to hold a zero factor, and why those?

In [ ]:
# your turn